### libraries

In [1]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split, cross_val_score,GridSearchCV
from sklearn.linear_model import ElasticNet, SGDRegressor, LinearRegression, Lasso, BayesianRidge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from xgboost.sklearn import XGBRegressor

### data preprocessing

In [2]:
df = pd.read_csv('laptop_price.csv', encoding='latin-1')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         1303 non-null   int64  
 1   Company           1303 non-null   object 
 2   Product           1303 non-null   object 
 3   TypeName          1303 non-null   object 
 4   Inches            1303 non-null   float64
 5   ScreenResolution  1303 non-null   object 
 6   Cpu               1303 non-null   object 
 7   Ram               1303 non-null   object 
 8   Gpu               1303 non-null   object 
 9   OpSys             1303 non-null   object 
 10  Weight            1303 non-null   object 
 11  Price_euros       1303 non-null   float64
dtypes: float64(2), int64(1), object(9)
memory usage: 122.3+ KB


In [3]:
df.describe()

,laptop_ID,Inches,Price_euros
count,1303.000000,1303.000000,1303.000000
mean,660.155794,15.017191,1123.686992
std,381.172104,1.426304,699.009043
min,1.000000,10.100000,174.000000
25%,331.500000,14.000000,599.000000
50%,659.000000,15.600000,977.000000
75%,990.500000,15.600000,1487.880000
max,1320.000000,18.400000,6099.000000


In [4]:
df = df.drop(['laptop_ID', 'Product'], axis=1)
df.head(3)

,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Gpu,OpSys,Weight,Price_euros
0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,Intel HD Graphics 620,No OS,1.86kg,575.00


In [5]:
df['Weight'] = df['Weight'].astype(str).str.replace('kg', '').astype(float)
df['Ram'] = df['Ram'].astype(str).str.replace('GB', '').astype(float)
df.head(3)

,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Gpu,OpSys,Weight,Price_euros
0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8.0,Intel Iris Plus Graphics 640,macOS,1.37,1339.69
1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8.0,Intel HD Graphics 6000,macOS,1.34,898.94
2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8.0,Intel HD Graphics 620,No OS,1.86,575.00


In [6]:
df.Company.value_counts()

Company
Dell         297
Lenovo       297
HP           274
Asus         158
Acer         103
MSI           54
Toshiba       48
Apple         21
Samsung        9
Razer          7
Mediacom       7
Microsoft      6
Xiaomi         4
Vero           4
Chuwi          3
Google         3
Fujitsu        3
LG             3
Huawei         2
Name: count, dtype: int64

In [7]:
df['Company'] = df['Company'].apply(lambda c: 'Other' if c not in ['Dell', 'Lenovo', 'HP', 'Asus', 'Acer', 'MSI', 'Toshiba', 'Apple'] else c)
df.Company.value_counts()

Company
Dell       297
Lenovo     297
HP         274
Asus       158
Acer       103
MSI         54
Other       51
Toshiba     48
Apple       21
Name: count, dtype: int64

In [8]:
df['TypeName'].value_counts()

TypeName
Notebook              727
Gaming                205
Ultrabook             196
2 in 1 Convertible    121
Workstation            29
Netbook                25
Name: count, dtype: int64

In [9]:
df['ScreenResolution'].value_counts()

ScreenResolution
Full HD 1920x1080                                507
1366x768                                         281
IPS Panel Full HD 1920x1080                      230
IPS Panel Full HD / Touchscreen 1920x1080         53
Full HD / Touchscreen 1920x1080                   47
1600x900                                          23
Touchscreen 1366x768                              16
Quad HD+ / Touchscreen 3200x1800                  15
IPS Panel 4K Ultra HD 3840x2160                   12
IPS Panel 4K Ultra HD / Touchscreen 3840x2160     11
4K Ultra HD / Touchscreen 3840x2160               10
4K Ultra HD 3840x2160                              7
Touchscreen 2560x1440                              7
IPS Panel 1366x768                                 7
IPS Panel Quad HD+ / Touchscreen 3200x1800         6
IPS Panel Retina Display 2560x1600                 6
IPS Panel Retina Display 2304x1440                 6
Touchscreen 2256x1504                              6
IPS Panel Touchscreen 2560x14

In [10]:
df['Resolution'] = df['ScreenResolution'].apply(lambda r : r.split(' ')[-1])
df['FullHD'] = df['ScreenResolution'].apply(lambda f : 1 if 'Full HD' in f else 0)
df['IpsPanel'] = df['ScreenResolution'].apply(lambda f : 1 if 'IPS Panel' in f else 0)
df['UltraHD'] = df['ScreenResolution'].apply(lambda f : 1 if 'Ultra HD' in f else 0)
df['TouchScreen'] = df['ScreenResolution'].apply(lambda f : 1 if 'Touchscreen' in f else 0)
df['QuadHD'] = df['ScreenResolution'].apply(lambda f : 1 if 'Quad HD' in f else 0)
df['RatinaDisplay'] = df['ScreenResolution'].apply(lambda f : 1 if 'Retina Display' in f else 0)

df = df.drop(['ScreenResolution'], axis=1)

df.head(3)

,Company,TypeName,Inches,Cpu,Ram,Gpu,OpSys,Weight,Price_euros,Resolution,FullHD,IpsPanel,UltraHD,TouchScreen,QuadHD,RatinaDisplay
0,Apple,Ultrabook,13.3,Intel Core i5 2.3GHz,8.0,Intel Iris Plus Graphics 640,macOS,1.37,1339.69,2560x1600,0,1,0,0,0,1
1,Apple,Ultrabook,13.3,Intel Core i5 1.8GHz,8.0,Intel HD Graphics 6000,macOS,1.34,898.94,1440x900,0,0,0,0,0,0
2,HP,Notebook,15.6,Intel Core i5 7200U 2.5GHz,8.0,Intel HD Graphics 620,No OS,1.86,575.00,1920x1080,1,0,0,0,0,0


In [11]:
df['Cpu'].value_counts()

Cpu
Intel Core i5 7200U 2.5GHz       190
Intel Core i7 7700HQ 2.8GHz      146
Intel Core i7 7500U 2.7GHz       134
Intel Core i7 8550U 1.8GHz        73
Intel Core i5 8250U 1.6GHz        72
                                ... 
Intel Core M M3-6Y30 0.9GHz        1
AMD A9-Series 9420 2.9GHz          1
Intel Core i3 6006U 2.2GHz         1
AMD A6-Series 7310 2GHz            1
Intel Xeon E3-1535M v6 3.1GHz      1
Name: count, Length: 118, dtype: int64

In [12]:
df['GHz'] = df['Cpu'].apply(lambda h : float(h.split(' ')[-1].replace('GHz', '')))
df['Processor'] = df['Cpu'].apply(lambda c : c.split(' ')[0])

df = df.drop(['Cpu'], axis=1)

df.head(3)

,Company,TypeName,Inches,Ram,Gpu,OpSys,Weight,Price_euros,Resolution,FullHD,IpsPanel,UltraHD,TouchScreen,QuadHD,RatinaDisplay,GHz,Processor
0,Apple,Ultrabook,13.3,8.0,Intel Iris Plus Graphics 640,macOS,1.37,1339.69,2560x1600,0,1,0,0,0,1,2.3,Intel
1,Apple,Ultrabook,13.3,8.0,Intel HD Graphics 6000,macOS,1.34,898.94,1440x900,0,0,0,0,0,0,1.8,Intel
2,HP,Notebook,15.6,8.0,Intel HD Graphics 620,No OS,1.86,575.00,1920x1080,1,0,0,0,0,0,2.5,Intel


In [13]:
df['Gpu'].value_counts(9)

Gpu
Intel HD Graphics 620      0.215656
Intel HD Graphics 520      0.141980
Intel UHD Graphics 620     0.052187
Nvidia GeForce GTX 1050    0.050652
Nvidia GeForce GTX 1060    0.036838
                             ...   
AMD Radeon R5 520          0.000767
AMD Radeon R7              0.000767
Intel HD Graphics 540      0.000767
AMD Radeon 540             0.000767
ARM Mali T860 MP4          0.000767
Name: proportion, Length: 110, dtype: float64

In [14]:
def graphics(x):
    s = x.split(' ')
    return ' '.join([s[0], s[1]])

df['Graphic'] = df['Gpu'].apply(graphics)
df = df.drop(['Gpu'], axis=1)

df.head(3)

,Company,TypeName,Inches,Ram,OpSys,Weight,Price_euros,Resolution,FullHD,IpsPanel,UltraHD,TouchScreen,QuadHD,RatinaDisplay,GHz,Processor,Graphic
0,Apple,Ultrabook,13.3,8.0,macOS,1.37,1339.69,2560x1600,0,1,0,0,0,1,2.3,Intel,Intel Iris
1,Apple,Ultrabook,13.3,8.0,macOS,1.34,898.94,1440x900,0,0,0,0,0,0,1.8,Intel,Intel HD
2,HP,Notebook,15.6,8.0,No OS,1.86,575.00,1920x1080,1,0,0,0,0,0,2.5,Intel,Intel HD


In [15]:
df['OpSys'].value_counts()

OpSys
Windows 10      1072
No OS             66
Linux             62
Windows 7         45
Chrome OS         27
macOS             13
Mac OS X           8
Windows 10 S       8
Android            2
Name: count, dtype: int64

In [16]:
df['OpSys'] = df['OpSys'].apply(lambda o : 'Other' if o in ['Chrome OS', 'macOS', 'Mac OS X', 'Windows 10 S', 'Android'] else o)
df['OpSys'].value_counts()

OpSys
Windows 10    1072
No OS           66
Linux           62
Other           58
Windows 7       45
Name: count, dtype: int64

In [17]:
df = pd.get_dummies(df)
df.head(5)

,Inches,Ram,Weight,Price_euros,FullHD,IpsPanel,UltraHD,TouchScreen,QuadHD,RatinaDisplay,...,Graphic_AMD R4,Graphic_AMD Radeon,Graphic_ARM Mali,Graphic_Intel Graphics,Graphic_Intel HD,Graphic_Intel Iris,Graphic_Intel UHD,Graphic_Nvidia GTX,Graphic_Nvidia GeForce,Graphic_Nvidia Quadro
0,13.3,8.0,1.37,1339.69,0,1,0,0,0,1,...,False,False,False,False,False,True,False,False,False,False
1,13.3,8.0,1.34,898.94,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
2,15.6,8.0,1.86,575.00,1,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
3,15.4,16.0,1.83,2537.45,0,1,0,0,0,1,...,False,True,False,False,False,False,False,False,False,False
4,13.3,8.0,1.37,1803.60,0,1,0,0,0,1,...,False,False,False,False,False,True,False,False,False,False


In [18]:
df.columns

Index(['Inches', 'Ram', 'Weight', 'Price_euros', 'FullHD', 'IpsPanel',
       'UltraHD', 'TouchScreen', 'QuadHD', 'RatinaDisplay', 'GHz',
       'Company_Acer', 'Company_Apple', 'Company_Asus', 'Company_Dell',
       'Company_HP', 'Company_Lenovo', 'Company_MSI', 'Company_Other',
       'Company_Toshiba', 'TypeName_2 in 1 Convertible', 'TypeName_Gaming',
       'TypeName_Netbook', 'TypeName_Notebook', 'TypeName_Ultrabook',
       'TypeName_Workstation', 'OpSys_Linux', 'OpSys_No OS', 'OpSys_Other',
       'OpSys_Windows 10', 'OpSys_Windows 7', 'Resolution_1366x768',
       'Resolution_1440x900', 'Resolution_1600x900', 'Resolution_1920x1080',
       'Resolution_1920x1200', 'Resolution_2160x1440', 'Resolution_2256x1504',
       'Resolution_2304x1440', 'Resolution_2400x1600', 'Resolution_2560x1440',
       'Resolution_2560x1600', 'Resolution_2736x1824', 'Resolution_2880x1800',
       'Resolution_3200x1800', 'Resolution_3840x2160', 'Processor_AMD',
       'Processor_Intel', 'Processor_Samsu

### model development

In [19]:
x = df.drop(['Price_euros'], axis=1)
y = df['Price_euros']

x.shape, y.shape

((1303, 60), (1303,))

In [20]:
x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.8)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((1042, 60), (261, 60), (1042,), (261,))

In [21]:
def model_acc(model):
    model.fit(x_train,y_train)
    acc = model.score(x_test, y_test)
    crossval = cross_val_score(model, x_test, y_test, cv=4).mean()
    print(f"train acc = {acc} & cross val score = {crossval} & model {str(model)}")

In [22]:
ElasticNet = ElasticNet()
SGDRegressor = SGDRegressor()
LinearRegression = LinearRegression()
Lasso = Lasso()
BayesianRidge = BayesianRidge()

model_acc(ElasticNet)
model_acc(SGDRegressor)
model_acc(LinearRegression)
model_acc(Lasso)
model_acc(BayesianRidge)

train acc = 0.7114107392819272 & cross val score = 0.6637932516564973 & model ElasticNet()
train acc = 0.6515659870345789 & cross val score = 0.5514648295612441 & model SGDRegressor()
train acc = 0.7767019609137976 & cross val score = 0.6789636098913439 & model LinearRegression()
train acc = 0.7794359833552771 & cross val score = 0.7063324992674751 & model Lasso()
train acc = 0.7764799623802503 & cross val score = 0.7225661431466642 & model BayesianRidge()


In [23]:
RandomForestRegressor = RandomForestRegressor()
GradientBoostingRegressor = GradientBoostingRegressor()
HistGradientBoostingRegressor = HistGradientBoostingRegressor()

model_acc(RandomForestRegressor)
model_acc(GradientBoostingRegressor)
model_acc(HistGradientBoostingRegressor)

train acc = 0.8413208484452672 & cross val score = 0.7490930816114953 & model RandomForestRegressor()
train acc = 0.8310197542713231 & cross val score = 0.7700138936136909 & model GradientBoostingRegressor()
train acc = 0.7802094293967176 & cross val score = 0.5848735018106127 & model HistGradientBoostingRegressor()


In [24]:
XGBRegressor = XGBRegressor()

model_acc(XGBRegressor)

train acc = 0.8426666770588029 & cross val score = 0.6778280626136353 & model XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)


### finetune model

In [25]:
param_grid = {'loss':['squared_error', 'absolute_error', 'huber', 'quantile'], 'learning_rate':[0.0, 0.01, 0.1, 0.2, 0.3], 'n_estimators': [200, 300, 400]}

grid_obj = GridSearchCV(estimator=GradientBoostingRegressor, param_grid=param_grid, cv=5, verbose=2)
grid_fit = grid_obj.fit(x_train, y_train)
best_model = grid_fit.best_estimator_
best_model.score(x_test, y_test)

Fitting 5 folds for each of 60 candidates, totalling 300 fits
[CV] END learning_rate=0.0, loss=squared_error, n_estimators=200; total time=   0.5s
[CV] END learning_rate=0.0, loss=squared_error, n_estimators=200; total time=   0.6s
[CV] END learning_rate=0.0, loss=squared_error, n_estimators=200; total time=   0.5s
[CV] END learning_rate=0.0, loss=squared_error, n_estimators=200; total time=   0.5s
[CV] END learning_rate=0.0, loss=squared_error, n_estimators=200; total time=   0.5s
[CV] END learning_rate=0.0, loss=squared_error, n_estimators=300; total time=   0.8s
[CV] END learning_rate=0.0, loss=squared_error, n_estimators=300; total time=   0.8s
[CV] END learning_rate=0.0, loss=squared_error, n_estimators=300; total time=   1.2s
[CV] END learning_rate=0.0, loss=squared_error, n_estimators=300; total time=   1.2s
[CV] END learning_rate=0.0, loss=squared_error, n_estimators=300; total time=   1.0s
[CV] END learning_rate=0.0, loss=squared_error, n_estimators=400; total time=   1.2s
[CV

0.8416667427565319

In [26]:
best_model

GradientBoostingRegressor(learning_rate=0.2, loss='huber', n_estimators=400)

### model saving

In [27]:
with open('predictor.pickle', 'wb') as file:
    pickle.dump(best_model,file)